<a href="https://colab.research.google.com/github/YUGOROU/amd-voice-sft/blob/feature%2Fdata-preprocessing/preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AMD Voice SFT — Data Preprocessing

Converts 3 HuggingFace datasets to ChatML format for SFT training.

**Datasets:**
- `fadodr/mental_health_therapy` (8,580 train rows, single-turn)
- `Estwld/empathetic_dialogues_llm` (19,500 train rows, multi-turn)
- `HuggingFaceTB/everyday-conversations-llama3.1-2k` (2,263 train rows, multi-turn)

**Output:** `data/train.jsonl`, `data/val.jsonl` (90:10 split)

In [1]:
!uv pip install -q datasets tqdm huggingface_hub

In [2]:
import json
import os
import random
from datasets import load_dataset
from tqdm import tqdm

SYSTEM_PROMPT = (
    "You are a caring and patient AI companion for elderly users with "
    "memory difficulties. Always respond with warmth, patience, and clarity."
)

SEED = 42
VAL_RATIO = 0.1

In [3]:
# --- Conversion functions ---

def convert_mental_health(ex):
    """fadodr/mental_health_therapy: instruction/input/output -> ChatML"""
    user_text = ex["input"].strip()
    assistant_text = ex["output"].strip()
    if not user_text or not assistant_text:
        return None
    return {"messages": [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_text},
        {"role": "assistant", "content": assistant_text},
    ]}


def convert_empathetic(ex):
    """Estwld/empathetic_dialogues_llm: conversations (role/content list) -> ChatML"""
    convs = ex["conversations"]
    if not convs or convs[0]["role"] != "user":
        return None
    return {"messages": [{"role": "system", "content": SYSTEM_PROMPT}] + convs}


def convert_everyday(ex):
    """HuggingFaceTB/everyday-conversations: messages (role/content list) -> ChatML"""
    msgs = ex["messages"]
    if not msgs or msgs[0]["role"] != "user":
        return None
    return {"messages": [{"role": "system", "content": SYSTEM_PROMPT}] + msgs}

In [4]:
# --- Load and convert ---

datasets_config = [
    ("fadodr/mental_health_therapy",                    "train",     convert_mental_health),
    ("Estwld/empathetic_dialogues_llm",                 "train", convert_empathetic),
    ("HuggingFaceTB/everyday-conversations-llama3.1-2k","train_sft", convert_everyday),
]

all_samples = []
source_counts = {}

for ds_id, split, fn in datasets_config:
    print(f"Loading {ds_id} [{split}]...")
    ds = load_dataset(ds_id, split=split)
    converted = [fn(ex) for ex in tqdm(ds, desc=ds_id.split("/")[1])]
    valid = [s for s in converted if s is not None]
    source_counts[ds_id] = {"raw": len(ds), "converted": len(valid)}
    all_samples.extend(valid)
    print(f"  {len(ds)} -> {len(valid)} samples")

print(f"\nTotal before filtering: {len(all_samples)}")

Loading fadodr/mental_health_therapy [train]...


README.md:   0%|          | 0.00/969 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/12.0M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/5.18M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8580 [00:00<?, ? examples/s]

...

data/test_sft-00000-of-00001.parquet:   0%|          | 0.00/124k [00:00<?, ?B/s]

Generating train_sft split:   0%|          | 0/2260 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/119 [00:00<?, ? examples/s]

everyday-conversations-llama3.1-2k: 100%|██████████| 2260/2260 [00:00<00:00, 4007.75it/s]

  2260 -> 2260 samples

Total before filtering: 30353


In [5]:
# --- Quality filter ---

def is_quality(sample):
    for m in sample["messages"]:
        if m["role"] == "user"      and len(m["content"]) < 20:
            return False
        if m["role"] == "assistant" and len(m["content"]) < 50:
            return False
    return True

before = len(all_samples)
all_samples = [s for s in all_samples if is_quality(s)]
print(f"Quality filter: {before} -> {len(all_samples)} ({len(all_samples)/before*100:.1f}% kept)")

Quality filter: 30353 -> 14365 (47.3% kept)


In [6]:
# --- Shuffle and split 90:10 ---

random.seed(SEED)
random.shuffle(all_samples)

split_idx = int(len(all_samples) * (1 - VAL_RATIO))
train_data = all_samples[:split_idx]
val_data   = all_samples[split_idx:]

print(f"Train: {len(train_data):,}  |  Val: {len(val_data):,}")

Train: 12,928  |  Val: 1,437


In [7]:
# --- Save as JSONL ---

os.makedirs("data", exist_ok=True)

for name, data in [("train", train_data), ("val", val_data)]:
    path = f"data/{name}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for s in data:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")
    print(f"Saved {path} ({len(data):,} samples)")

# Spot-check first sample
print("\n--- Sample train[0] ---")
print(json.dumps(train_data[0], indent=2, ensure_ascii=False))

Saved data/train.jsonl (12,928 samples)
Saved data/val.jsonl (1,437 samples)

--- Sample train[0] ---
{
  "messages": [
    {
      "role": "system",
      "content": "You are a caring and patient AI companion for elderly users with memory difficulties. Always respond with warmth, patience, and clarity."
    },
    {
      "content": "The other day I was getting my mail and my neighbors little grandson accidentally let the little devil chihuahua out! I dont think I have ever run so fast in my life! ",
      "role": "user"
    },
    {
      "content": "Oh no! They are terrifying little dogs aren't they?",
      "role": "assistant"
    },
    {
      "content": "Yes! Their bites hurt so bad and they are so flipping fast!",
      "role": "user"
    },
    {
      "content": "They're really loud too actually, they bark a whole lot more than other dogs",
      "role": "assistant"
    }
  ]
}


In [8]:
# --- Upload to Hugging Face Hub ---
# Uses HF_TOKEN from Colab Secrets (left panel > key icon > add HF_TOKEN)

from google.colab import userdata
from huggingface_hub import HfApi

hf_token = userdata.get("HF_TOKEN")
api = HfApi(token=hf_token)

api.create_repo(
    repo_id="YUGOROU/amd-voice-sft-data",
    repo_type="dataset",
    private=True,
    exist_ok=True,
)
api.upload_folder(
    folder_path="data",
    repo_id="YUGOROU/amd-voice-sft-data",
    repo_type="dataset",
)
print("Uploaded to HF Hub: YUGOROU/amd-voice-sft-data")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/data/train.jsonl   :   4%|4         | 1.10MB / 26.7MB            

Uploaded to HF Hub: YUGOROU/amd-voice-sft-data
